In [1]:
from langchain.tools import tool
from ddgs import DDGS
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent

In [2]:
@tool
def web_search(query: str) -> str:
    """Search the web for current information."""
    
    results = DDGS().text(query, max_results=5)

    return "\n".join(
        f"- {item['title']}: {item['body']}"
        for item in results
    )

In [3]:
print(web_search.invoke("latest developments in agentic AI"))

- 7 Agentic AI Trends to Watch in 2026 - MachineLearningMastery.com: January 5, 2026 - Discover the seven emerging trends reshaping agentic AI in 2026, from multi-agent orchestration to production scaling challenges.
- Agentic AI: 4 reasons why it’s the next big thing in AI research | IBM: November 18, 2025 - Learn how leading enterprises orchestrate, build and govern agentic AI with an open, hybrid approach to move from experimentation to real impact. ... One can imagine many business functions currently performed with software as a service (SaaS) products being replaced or augmented by agentic systems, which enable workers to interact with data and perform tasks more efficiently with natural language inputs and simplified user interfaces. For example, imagine a ticketing system that software developers use to track the progress of projects.
- Latest Agentic AI News Today | Trends, Predictions, & Analysis: 2 weeks ago - DaVinci Commerce and Accenture are building agentic commerce infr

In [2]:
from dotenv import load_dotenv
import os
from pathlib import Path

env_path = r"C:\Users\abhin\OneDrive\Desktop\openai\.env"

print("File exists:", Path(env_path).exists())

loaded = load_dotenv(env_path)
print("Loaded:", loaded)

print("Key found:", bool(os.getenv("OPENAI_API_KEY")))

File exists: True
Loaded: True
Key found: True


In [3]:
import os
from dotenv import load_dotenv
from ddgs import DDGS

from langchain.agents import create_agent
from langchain.tools import tool
from langchain_openai import ChatOpenAI

# Load OPENAI_API_KEY from your .env file
load_dotenv()

print("API key loaded:", bool(os.getenv("OPENAI_API_KEY")))


# Custom tool
@tool
def simple_tool(query: str) -> str:
    """Return a simple response for a user query."""
    return f"Tool response: {query}"


# Web-search tool
@tool
def web_search(query: str) -> str:
    """Search the web for current information such as weather, news, or recent events."""
    results = list(DDGS().text(query, max_results=5))

    if not results:
        return "No search results found."

    return "\n".join(
        f"- {item.get('title', 'Result')}: {item.get('body', '')}"
        for item in results
    )


# Model
llm = ChatOpenAI(
    model="gpt-4.1-mini",
    temperature=0
)

# Create the agent
agent = create_agent(
    model=llm,
    tools=[web_search, simple_tool],
    system_prompt="""
    You are a helpful assistant.
    Always use web_search for current information such as weather, news, prices, or recent events.
    """
)

# Ask the agent
result = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "What's the weather like today in London?"
            }
        ]
    }
)

print(result["messages"][-1].content)

API key loaded: True
Today's weather in London on June 30, 2024, is reported to have passing clouds with a humidity of 86%. Unfortunately, I don't have the exact temperature or other detailed current conditions at this moment. Would you like me to find more specific details like temperature, wind, or precipitation?


In [6]:
from langchain.messages import AIMessage, ToolMessage

question = "What's the weather like today in London?"

for update in agent.stream(
    {
        "messages": [
            {"role": "user", "content": question}
        ]
    },
    stream_mode="updates"
):
    for step_name, data in update.items():
        for message in data.get("messages", []):

            # Model wants to call a tool
            if isinstance(message, AIMessage) and message.tool_calls:
                for tool_call in message.tool_calls:
                    print("\nACTION")
                    print("Tool:", tool_call["name"])
                    print("Input:", tool_call["args"])

            # Tool returned its result
            elif isinstance(message, ToolMessage):
                print("\nOBSERVATION")
                print(message.content)

            # Final model answer
            elif isinstance(message, AIMessage) and message.content:
                print("\nFINAL ANSWER")
                print(message.content)


ACTION
Tool: web_search
Input: {'query': 'weather today in London'}

OBSERVATION
- Today's Weather in London - Hourly Forecast and Conditions: Get the latest hourly weather updates for London today. Detailed forecast including temperature, wind, rain, snow, and UV index. Stay informed about today's weather conditions in London.
- London - BBC Weather: 14-day weather forecast for London. Tonight Tonight will stay dry throughout with plenty of clear spells for all and just a few areas of patchy cloud at times.
- London, London, United Kingdom Hourly Weather | AccuWeather: Hourly weather forecast in London, London, United Kingdom. Check current conditions in London, London, United Kingdom with radar, hourly, and more.
- London (Greater London) weather - Met Office: London 7 day weather forecast including weather warnings, temperature, rain, wind, visibility, humidity and UV
- London local weather (live): today, hourly weather: Latest weather forecast for London for today's, hourly weathe

In [7]:
import os
from dotenv import load_dotenv
from ddgs import DDGS

from langchain.tools import tool
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI

load_dotenv()

print("API key loaded:", bool(os.getenv("OPENAI_API_KEY")))

# Current OpenAI model for practice
llm = ChatOpenAI(
    model="gpt-4.1-mini",
    temperature=0
)

# Prompt + model chain for summarizing text
summary_prompt = PromptTemplate.from_template(
    """
    Summarize the following content in simple bullet points.
    Include only the most important information.

    Content:
    {content}
    """
)

summary_chain = summary_prompt | llm


@tool
def ddg_search(query: str) -> str:
    """Search the web for current information."""
    results = list(DDGS().text(query, max_results=5))

    if not results:
        return "No search results found."

    return "\n\n".join(
        f"Title: {item.get('title', 'Result')}\n"
        f"Content: {item.get('body', '')}"
        for item in results
    )


@tool
def summarize_text(content: str) -> str:
    """Summarize text, an article, or web-search content into concise bullet points."""
    response = summary_chain.invoke({"content": content})
    return response.content

API key loaded: True


In [8]:
from langchain.agents import create_agent

agent = create_agent(
    model=llm,
    tools=[ddg_search, summarize_text],
    system_prompt="""
    You are a helpful research assistant.

    For current or factual topics, use ddg_search first.
    When the search result is long, use summarize_text before giving the final answer.
    Clearly state the final answer in simple language.
    """
)

In [9]:
question = "Search the web and summarize how to become a good footballer."

result = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": question
            }
        ]
    }
)

print(result["messages"][-1].content)

To become a good footballer, you need to practice consistently and focus on the fundamentals like passing, dribbling, and shooting. If you don't have a training partner, use a wall to practice ball control and passing drills. Work on improving your agility, speed, and footwork through specific drills. Structured practice and repetition are key to refining your skills. Your training should cover technical skills, tactical understanding, and physical conditioning. Most importantly, enjoy the game and be patient with your progress. There are many drills available for all skill levels to help improve different aspects of your game.


In [10]:
from langchain_core.messages import AIMessage, ToolMessage

question = "Search the web and summarize how to become a good footballer."

def short_text(value, limit=1200):
    """Keep long search results readable in Jupyter."""
    text = str(value)
    return text if len(text) <= limit else text[:limit] + "\n... [truncated]"


print("> Entering agent workflow...\n")

for update in agent.stream(
    {
        "messages": [
            {"role": "user", "content": question}
        ]
    },
    stream_mode="updates"
):
    for step_name, data in update.items():

        for message in data.get("messages", []):

            # The model selected one or more tools
            if isinstance(message, AIMessage) and message.tool_calls:
                print("PLAN / ACTION")
                print(f"Agent step: {step_name}")

                for tool_call in message.tool_calls:
                    print("Tool:", tool_call["name"])
                    print("Input:", tool_call["args"])

                print("-" * 70)

            # A tool returned its result
            elif isinstance(message, ToolMessage):
                print("OBSERVATION")
                print("Tool:", message.name or "tool")
                print(short_text(message.content))
                print("-" * 70)

            # The model produced the final user-facing answer
            elif isinstance(message, AIMessage) and message.content:
                print("FINAL ANSWER")
                print(short_text(message.content))
                print("-" * 70)

print("> Finished agent workflow.")

> Entering agent workflow...

PLAN / ACTION
Agent step: model
Tool: ddg_search
Input: {'query': 'how to become a good footballer'}
----------------------------------------------------------------------
OBSERVATION
Tool: ddg_search
Title: How to Become a Better Footballer | 6 Key Steps
Content: How to Become a Better Footballer · 1. Master the Basics · 2. Build Fitness and Strength · 3. Develop Game Awareness · 4. Train with Purpose · 5. Stay Mentally ...

Title: A Guide to being a Good Football Player. - Reddit
Content: Aug 20, 2023 ... If you want to be a good football player, focus on mastering fundamentals, maintaining peak physical fitness, and cultivating teamwork and ...

Title: How To Train Like a Professional Footballer - YouTube
Content: Mar 16, 2026 ... ... football training, how to become a pro footballer, pro level football training drills, drills for professional footballers. How To Train Like ...

Title: How to Become a Better Soccer Player: Essential Skills and Tips
Cont